In [2]:
%%configure
{
    "defaultLakehouse": {
        "name": "lh_oyster_path"
    }
}

StatementMeta(, a4167e5c-85e8-4c01-a3fa-a0861fb129ea, -1, Finished, Available, Finished, True)

In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import functions as F




StatementMeta(, 76f0632f-5d0b-4c11-893f-d0a1f5928c76, 3, Finished, Available, Finished, False)

In [2]:
# Define the relative path using wildcards to capture all date folders
# This targets all 'Customer' CSV files across any date folder inside 'oyster'
p_root_path = f"Files/landing/oyster/"
p_in_table_name = "Transaction"

StatementMeta(, 76f0632f-5d0b-4c11-893f-d0a1f5928c76, 4, Finished, Available, Finished, False)

In [3]:
target_table = f"dbo.ld_oys_{p_in_table_name}".lower()

StatementMeta(, 76f0632f-5d0b-4c11-893f-d0a1f5928c76, 5, Finished, Available, Finished, False)

In [4]:

# Build the recursive glob pattern
pattern = f"{p_root_path.rstrip('/')}/**/{p_in_table_name}_*.csv"

# Extract only the file paths (keys) to avoid loading file contents into driver memory
source_files = spark.sparkContext.binaryFiles(pattern).keys().collect()

if not source_files:
    raise FileNotFoundError(f"No raw files match {pattern}")

# Sort paths chronologically by the yyyymmddhhmmss folder name and pick the last one
# Note: Extracting the parent folder name (-2) ensures sorting is strictly based on the folder
latest_file_path = sorted(source_files, key=lambda path: path.split("/")[-2])[-1]

print(f"Matched {len(source_files)} file(s).")
print(f"Latest file selected: {latest_file_path}")


StatementMeta(, 76f0632f-5d0b-4c11-893f-d0a1f5928c76, 6, Finished, Available, Finished, False)

Matched 1 file(s).
Latest file selected: abfss://5e32bf37-c2be-4676-94c7-97213238ed30@onelake.dfs.fabric.microsoft.com/f27d086e-be7b-4e40-8e62-ea4d210b4538/Files/landing/oyster/20261109142245/Transaction_Ledger_20261109T142245.csv


In [5]:
# Read only the latest CSV file
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(latest_file_path)
)

# Clean up the file name (removes '?version=...' suffix) and add metadata columns
raw_file_name = F.element_at(F.split(F.input_file_name(), "/"), -1)
clean_file_name = F.element_at(F.split(raw_file_name, "\\?"), 1)

df_with_meta = (
    df.withColumn("meta_load_timestamp_bp", F.from_utc_timestamp(F.current_timestamp(), "Europe/Budapest"))
      .withColumn("meta_load_timestamp_kyiv", F.from_utc_timestamp(F.current_timestamp(), "Europe/Kyiv"))
      .withColumn("meta_load_timestamp_chisinau", F.from_utc_timestamp(F.current_timestamp(), "Europe/Chisinau"))
      .withColumn("meta_source_file", clean_file_name)
)

# Overwrite the managed Delta table
(
    df_with_meta.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Successfully loaded '{latest_file_path} ' into {target_table}")


StatementMeta(, 76f0632f-5d0b-4c11-893f-d0a1f5928c76, 7, Finished, Available, Finished, False)

Successfully loaded 'abfss://5e32bf37-c2be-4676-94c7-97213238ed30@onelake.dfs.fabric.microsoft.com/f27d086e-be7b-4e40-8e62-ea4d210b4538/Files/landing/oyster/20261109142245/Transaction_Ledger_20261109T142245.csv ' into dbo.ld_oys_transaction___


In [6]:
df = spark.sql(f"""SELECT 
    meta_load_timestamp,
    meta_source_file,
    COUNT(*) AS row_count
FROM {target_table}
GROUP BY meta_load_timestamp, meta_source_file
ORDER BY meta_source_file DESC""")

display(df)


StatementMeta(, 76f0632f-5d0b-4c11-893f-d0a1f5928c76, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0572f9ea-d33e-4f3a-980e-e6faae512042)